### Pré-processamento do BETA dataset

Nesta etapa, os dados do participante S001 são organizados no formato que o MNE espera, `(n_epochs, n_channels, n_times)`, e o objeto `info` é criado.

In [45]:
# Carregar os dados

import numpy as np

data = np.load('../data/output/S001_pre_pros.npy')

print(data.dtype) # float64
print(data.shape)

channels_index, time_points, electrode_index, block_index, target_index = data.shape
channels = ['POz', 'PO3', 'PO4', 'PO5', 'PO6', 'Oz', 'O1', 'O2']
fs = 250

float64
(8, 710, 2, 10, 12)


**Monta o array data `[canal, tempo, eletrodo, bloco, alvo]` na forma esperada pelo MNE [n_epochs, n_channels, n_times]**

Para isso, cada par (bloco, alvo) corresponde a uma epoch no formato [8, 710]. Como cada conjunto tem 10 blocos e 12 alvos, totalizam 120 epochs. Os tipos de eletrodos `dry` e `wet` foram separados em dois conjuntos, por serem duas toucas diferentes. Por fim, os labels guardam o índice do alvo de 0 a 11 de cada epoch.

In [46]:
import mne

# Move a ordem do array e faz um reshape

epochs_all = data.transpose(2, 3, 4, 0, 1)
print(epochs_all.shape) # [eletrodo, bloco, alvo, canal, tempo]

epochs_all = epochs_all.reshape(electrode_index * block_index * target_index, channels_index, time_points)
print(epochs_all.shape)

(2, 10, 12, 8, 710)
(240, 8, 710)


In [47]:
# Separar entre touca com eletrodos molhados e secos

number_of_epochs_per_electrode = block_index * target_index

epochs_dry = epochs_all[0:number_of_epochs_per_electrode]
epochs_wet = epochs_all[number_of_epochs_per_electrode:]

print(epochs_dry.shape)
print(epochs_wet.shape)

(120, 8, 710)
(120, 8, 710)


**Verificação**

Como cada época deve ser idêntica ao trecho correspondente do array original, o trecho abaixo compara a época 0 (bloco 0, alvo 0) e a época 13 (bloco 1, alvo 1). Como os loops percorrem os blocos e, dentro de cada bloco, os alvos, a época de índice `b * 12 + t` corresponde a `(bloco b, alvo t)`.

In [48]:
target_labels = np.tile(np.arange(target_index), block_index)

print(np.array_equal(epochs_dry[0], data[:, :, 0, 0, 0]))
print(np.array_equal(epochs_dry[13], data[:, :, 0, 1, 1]))
print(np.array_equal(epochs_wet[13], data[:, :, 1, 1, 1]))
print(target_labels[0:24])
print(target_labels.shape)

True
True
True
[ 0  1  2  3  4  5  6  7  8  9 10 11  0  1  2  3  4  5  6  7  8  9 10 11]
(120,)


**Objeto info**

In [49]:
sfreq = fs
ch_names = channels

info = mne.create_info(ch_names, sfreq=sfreq, ch_types='eeg')
info

<Info | 7 non-empty values
 bads: []
 ch_names: POz, PO3, PO4, PO5, PO6, Oz, O1, O2
 chs: 8 EEG
 custom_ref_applied: False
 highpass: 0.0 Hz
 lowpass: 125.0 Hz
 meas_date: unspecified
 nchan: 8
 projs: []
 sfreq: 250.0 Hz
>

**Implementação dos filtros**

In [50]:
from scipy import signal

def butter_bandpass(data, lowcut, highcut, fs=512, order=4):
    nyq = fs * 0.5
    low = lowcut / nyq
    high = highcut / nyq
    b, a = signal.butter(order, [low, high], btype='bandpass')
    return signal.filtfilt(b, a, data)


def butter_lowpass(data, lowcut, fs=512, order=4):
    nyq = fs * 0.5
    low = lowcut / nyq
    b, a = signal.butter(order, low, btype='lowpass')
    return signal.filtfilt(b, a, data)


def butter_highpass(data, highcut, fs=512, order=4):
    nyq = fs * 0.5
    high = highcut / nyq
    b, a = signal.butter(order, high, btype='highpass')
    return signal.filtfilt(b, a, data)


def butter_notch(data, cutoff, var=1, fs=512, order=4):
    nyq = fs * 0.5
    low = (cutoff - var) / nyq
    high = (cutoff + var) / nyq
    b, a = signal.iirfilter(order, [low, high], btype='bandstop', ftype="butter")
    return signal.filtfilt(b, a, data)


# Aplicando no BETA, com fs=250
epochs_dry_notch = butter_notch(epochs_dry, cutoff=50, fs=fs)
epochs_dry_filtered = butter_bandpass(epochs_dry_notch, lowcut=6, highcut=90, fs=fs)

print(epochs_dry_filtered.shape)

epochs_wet_notch = butter_notch(epochs_wet, cutoff=50, fs=fs)
epochs_wet_filtered = butter_bandpass(epochs_wet_notch, lowcut=6, highcut=90, fs=fs)

print(epochs_wet_filtered.shape)

(120, 8, 710)
(120, 8, 710)


**Visualizações:**

Visualização do canal Oz (eletrodo seco) nos três domínios: tempo, frequência (PSD) e espectrograma, sem a aplicação de nenhum filtro. São as 12 epochs do bloco 0, uma por alvo.

In [51]:
import matplotlib.pyplot as plt
from matplotlib import rcParams

def print_graphs(data):
    for i in range():
        plt.plot(data[i,:])
    plt.title('Domínio do tempo')
    plt.show()

    for i in range():
        plt.psd(data[i,:], Fs=fs)
    plt.title('Domínio da frequência')
    plt.show()

    for i in range():
        plt.specgram(data[i,:], Fs=fs)
    plt.title('Espectrograma, alvo ' + str(i))
    plt.show()